# EDA — Monza 2025: NOR Fastest Lap
Track map and telemetry channel plots.

In [1]:
import pathlib
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

ROOT = pathlib.Path('..').resolve()
DATA_DIR = ROOT / 'data'
FIG_DIR = ROOT / 'notebooks' / 'figures'
FIG_DIR.mkdir(exist_ok=True)

CSV = DATA_DIR / '2025_monza_NOR_fastest_lap_fused.csv'
df = pd.read_csv(CSV)
print(df.shape)
df.head(3)

Matplotlib is building the font cache; this may take a moment.


(602, 10)


,SessionTime_s,Distance,X,Y,Speed,RPM,Throttle,Brake,TrackTemp,AirTemp
0,7812.743,0.046865,-1383.396314,-799.762786,317.059999,11218.659986,99.0,0,40.3,27.4
1,7812.763,1.814291,-1382.000000,-782.000000,317.259999,11220.859986,99.0,0,40.3,27.4
2,7812.837,8.362222,-1376.836606,-715.842823,318.000000,11229.000000,99.0,0,40.3,27.4


## Plot 1 — Telemetry channels vs Distance

In [2]:
DRIVER = 'NOR'
LAP_NUM = 53
LAP_TIME = '1:20.901'

channels = [
    ('Speed',    'Speed (km/h)',  '#e63946'),
    ('Throttle', 'Throttle (%)',  '#2a9d8f'),
    ('Brake',    'Brake',         '#e9c46a'),
    ('RPM',      'RPM',           '#457b9d'),
]

fig, axes = plt.subplots(len(channels), 1, figsize=(14, 9), sharex=True)
fig.suptitle(f'{DRIVER} – Lap {LAP_NUM} ({LAP_TIME}) | 2025 Italian GP', fontsize=13, y=1.01)

for ax, (col, ylabel, color) in zip(axes, channels):
    ax.plot(df['Distance'], df[col], color=color, linewidth=0.9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

axes[-1].set_xlabel('Distance (m)', fontsize=10)
fig.tight_layout()
out1 = FIG_DIR / 'telemetry_channels.png'
fig.savefig(out1, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved → {out1}')

Saved → /Users/mattjanasiv/f1-lattice-twin/notebooks/figures/telemetry_channels.png


## Plot 2 — Track map coloured by Speed

In [3]:
x = df['X'].values
y = df['Y'].values
speed = df['Speed'].values

# Build line segments for smooth colouring
from matplotlib.collections import LineCollection

points = np.column_stack([x, y]).reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)
speed_mid = (speed[:-1] + speed[1:]) / 2

norm = mcolors.Normalize(vmin=speed.min(), vmax=speed.max())
cmap = plt.get_cmap('plasma')

lc = LineCollection(segments, cmap=cmap, norm=norm, linewidth=2.5, capstyle='round')
lc.set_array(speed_mid)

fig, ax = plt.subplots(figsize=(10, 8))
ax.add_collection(lc)
ax.set_xlim(x.min() - 50, x.max() + 50)
ax.set_ylim(y.min() - 50, y.max() + 50)
ax.set_aspect('equal')
ax.set_xticks([])
ax.set_yticks([])
ax.set_facecolor('#0d0d0d')
fig.patch.set_facecolor('#0d0d0d')

cbar = fig.colorbar(lc, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Speed (km/h)', color='white', fontsize=10)
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

ax.set_title(
    f'{DRIVER} – Lap {LAP_NUM} ({LAP_TIME}) | 2025 Italian GP\nColoured by Speed',
    color='white', fontsize=12, pad=12
)

out2 = FIG_DIR / 'track_map_speed.png'
fig.savefig(out2, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close(fig)
print(f'Saved → {out2}')

Saved → /Users/mattjanasiv/f1-lattice-twin/notebooks/figures/track_map_speed.png
